In [5]:
from langchain_ollama.llms import OllamaLLM
import os
import requests
import time
# from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./HP1RUS.txt")
documents = loader.load()
# ============================================================
# ШАГ 5: РАЗБИЕНИЕ НА ЧАНКИ
# ============================================================
print("\n🧩 Разбиение на чанки...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = text_splitter.split_documents(documents)
print(f"✅ Получено {len(splits)} чанков")



🧩 Разбиение на чанки...
✅ Получено 554 чанков


In [7]:
!pip install tqdm


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Установите sentence-transformers
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

class RuBERTEmbeddings(Embeddings):
    def __init__(self, model_name='deepvk/USER-bge-m3'):
        self.model = SentenceTransformer(model_name)
    
    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    
    def embed_query(self, text):
        return self.model.encode(text, normalize_embeddings=True).tolist()

# Вместо OllamaEmbeddings
# embeddings = OllamaEmbeddings(model="nomic-embed-text")
embeddings = RuBERTEmbeddings('deepvk/USER-bge-m3')

# Дальше ваш код без изменений
loader = TextLoader("./HP1RUS.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # Увеличил
    chunk_overlap=400,
)
splits = text_splitter.split_documents(documents)



vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,  # RuBERT
    persist_directory="./chroma_db_rag"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11029.70it/s]


KeyboardInterrupt: 

In [10]:
import torch
print("CUDA доступна:", torch.cuda.is_available())
print("Количество GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Имя вашей видеокарты:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU НЕ используется. Работаете на процессоре (CPU).")

CUDA доступна: False
Количество GPU: 0
❌ GPU НЕ используется. Работаете на процессоре (CPU).


In [ ]:
pip install sentence-transformers